In [1]:
!pip install -q librosa==0.10.2 lightgbm soundfile tqdm scikit-learn --upgrade
import warnings
warnings.filterwarnings("ignore")
print("Done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 41.1 MB/s eta 0:00:00
Done.


In [2]:
import os

if not os.path.exists("/content/ESC-50-master"):
    !wget -q https://github.com/karoldvl/ESC-50/archive/master.zip -O /content/esc50.zip
    !unzip -q /content/esc50.zip -d /content/
    print("Downloaded and extracted.")
else:
    print("Already present.")

AUDIO_DIR = "/content/ESC-50-master/audio"
META_PATH = "/content/ESC-50-master/meta/esc50.csv"
assert os.path.exists(META_PATH), "Metadata not found — check the download step."
print("Files:", len(os.listdir(AUDIO_DIR)))

Downloaded and extracted.
Files: 2000


In [3]:
SAMPLE_RATE = 22050
N_MELS = 64
DURATION = 5.0  # ESC-50 clips are all 5 seconds
FIXED_LEN = int(SAMPLE_RATE * DURATION)

# ── Category mapping — copied verbatim from modules/audio/esc50_loader.py ──
CATEGORY_MAP = {
    # Closest available proxy for quiet human activity / ambient presence
    "breathing":        "ambient",
    "coughing":          "ambient",
    "footsteps":         "ambient",
    "drinking_sipping":  "ambient",

    # Closest available proxy for "paper"-like rustling/handling sounds
    "keyboard_typing":   "paper",
    "mouse_click":       "paper",
    "door_wood_creaks":  "paper",

    # Closest available proxy for louder, disruptive sound
    "clapping":          "loud",
    "laughing":          "loud",
    "crying_baby":       "loud",
    "sneezing":          "loud",
    "door_wood_knock":   "loud",

    # True silence / background proxy
    "rain":              "silence",
    "wind":               "silence",
    "sea_waves":          "silence",
}

TARGET_CLASSES = ["silence", "ambient", "paper", "loud"]

RANDOM_STATE = 42
DRIVE_SAVE_DIR = "/content/drive/MyDrive/TrueWatch_models"  # set None to skip Drive save

In [4]:
if DRIVE_SAVE_DIR is not None:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    print("Models will be saved to:", DRIVE_SAVE_DIR)

Mounted at /content/drive
Models will be saved to: /content/drive/MyDrive/TrueWatch_models


In [5]:
import pandas as pd

def load_metadata():
    df = pd.read_csv(META_PATH)
    df = df[df["category"].isin(CATEGORY_MAP.keys())].copy()
    df["target_class"] = df["category"].map(CATEGORY_MAP)
    print(f"Selected {len(df)} clips across {df['category'].nunique()} "
          f"ESC-50 categories, mapped to {df['target_class'].nunique()} target classes")
    print(df["target_class"].value_counts())
    return df

meta = load_metadata()

missing = set(CATEGORY_MAP.keys()) - set(pd.read_csv(META_PATH)["category"].unique())
if missing:
    raise ValueError(f"These source categories aren't in ESC-50: {missing}")

Selected 600 clips across 15 ESC-50 categories, mapped to 4 target classes
target_class
loud       200
ambient    160
paper      120
silence    120
Name: count, dtype: int64


In [6]:
import numpy as np
import librosa

def load_audio_fixed(path, sr=SAMPLE_RATE, fixed_len=FIXED_LEN):
    y, _ = librosa.load(path, sr=sr, duration=DURATION)
    if len(y) < fixed_len:
        y = np.pad(y, (0, fixed_len - len(y)))
    else:
        y = y[:fixed_len]
    return y

def extract_mel_spectrogram_from_array(y, sr=SAMPLE_RATE, n_mels=N_MELS):
    """Same transform as esc50_loader.extract_mel_spectrogram, but takes an
    already-loaded array so it can be reused on augmented clips too."""
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels, fmax=8000, hop_length=512, n_fft=2048
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_norm.astype(np.float32)

def handcrafted_features(y, sr=SAMPLE_RATE):
    feats = []
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)

    for feat in [mfcc, chroma, contrast, tonnetz, zcr, rms, centroid, bandwidth, rolloff]:
        feats.append(feat.mean(axis=1))
        feats.append(feat.std(axis=1))
    return np.concatenate(feats).astype(np.float32)

print("Feature functions ready.")

Feature functions ready.


In [7]:
def augment_pitch(y, sr=SAMPLE_RATE):
    n_steps = np.random.uniform(-2, 2)
    return librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)

def augment_stretch(y):
    rate = np.random.uniform(0.85, 1.15)
    y_s = librosa.effects.time_stretch(y=y, rate=rate)
    if len(y_s) < FIXED_LEN:
        y_s = np.pad(y_s, (0, FIXED_LEN - len(y_s)))
    else:
        y_s = y_s[:FIXED_LEN]
    return y_s

def augment_noise(y, snr_db=20):
    rms_signal = np.sqrt(np.mean(y ** 2))
    rms_noise = rms_signal / (10 ** (snr_db / 20))
    noise = np.random.normal(0, rms_noise, y.shape)
    return (y + noise).astype(np.float32)

def augment_shift(y, max_shift=0.3):
    shift = int(np.random.uniform(-max_shift, max_shift) * SAMPLE_RATE)
    return np.roll(y, shift)

AUGMENTERS = [augment_pitch, augment_stretch, augment_noise, augment_shift]
print("Augmenters ready:", [f.__name__ for f in AUGMENTERS])

Augmenters ready: ['augment_pitch', 'augment_stretch', 'augment_noise', 'augment_shift']


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

le = LabelEncoder()
le.fit(TARGET_CLASSES)

paths = [os.path.join(AUDIO_DIR, f) for f in meta["filename"]]
labels = le.transform(meta["target_class"])

train_paths, test_paths, y_train_raw, y_test = train_test_split(
    paths, labels, test_size=0.2, random_state=RANDOM_STATE, stratify=labels
)
print(f"Train clips (pre-augmentation): {len(train_paths)}  |  Test clips: {len(test_paths)}")

Train clips (pre-augmentation): 480  |  Test clips: 120


In [9]:
def build_split(paths, labels, augment=False, n_aug=4):
    logmels, hand_feats, ys = [], [], []
    for path, label in tqdm(list(zip(paths, labels)), desc="Extracting"):
        y = load_audio_fixed(path)

        logmels.append(extract_mel_spectrogram_from_array(y))
        hand_feats.append(handcrafted_features(y))
        ys.append(label)

        if augment:
            chosen = np.random.choice(len(AUGMENTERS), size=n_aug, replace=True)
            for idx in chosen:
                y_aug = AUGMENTERS[idx](y.copy())
                logmels.append(extract_mel_spectrogram_from_array(y_aug))
                hand_feats.append(handcrafted_features(y_aug))
                ys.append(label)

    X_mel = np.stack(logmels)[..., np.newaxis]   # (N, n_mels, T, 1)
    X_hand = np.stack(hand_feats)                 # (N, n_hand_features)
    y_out = np.array(ys)
    return X_mel, X_hand, y_out

X_train_mel, X_train_hand, y_train = build_split(train_paths, y_train_raw, augment=True, n_aug=4)
X_test_mel,  X_test_hand,  y_test_arr = build_split(test_paths, y_test, augment=False)

print("Train mel:", X_train_mel.shape, " Train hand:", X_train_hand.shape)
print("Test mel: ", X_test_mel.shape,  " Test hand: ", X_test_hand.shape)

Extracting:   0%|          | 0/480 [00:00<?, ?it/s]

Extracting:   0%|          | 0/120 [00:00<?, ?it/s]

Train mel: (2400, 64, 216, 1)  Train hand: (2400, 100)
Test mel:  (120, 64, 216, 1)  Test hand:  (120, 100)


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_hand_s = scaler.fit_transform(X_train_hand)
X_test_hand_s = scaler.transform(X_test_hand)

In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models

n_classes = len(TARGET_CLASSES)
input_shape = X_train_mel.shape[1:]

def build_cnn(input_shape, n_classes):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

cnn = build_cnn(input_shape, n_classes)
cnn.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 216, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 216, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 216, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 108, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 108, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 108, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 108, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 54, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 54, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 27, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 27, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 27, 256)     │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 8, 27, 256)     │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 456,580 (1.74 MB)

 Trainable params: 455,620 (1.74 MB)

 Non-trainable params: 960 (3.75 KB)

In [12]:
from sklearn.utils.class_weight import compute_class_weight

classes_present = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes_present, y=y_train)
class_weight = dict(zip(classes_present, weights))
print("Class weights:", class_weight)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10,
                                      restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=5, min_lr=1e-6),
]

history = cnn.fit(
    X_train_mel, y_train,
    validation_split=0.15,
    epochs=60,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

Class weights: {np.int64(0): np.float64(0.9375), np.int64(1): np.float64(0.75), np.int64(2): np.float64(1.25), np.int64(3): np.float64(1.25)}
Epoch 1/60
64/64 ━━━━━━━━━━━━━━━━━━━━ 24s 191ms/step - accuracy: 0.5020 - loss: 1.1262 - val_accuracy: 0.2083 - val_loss: 4.9910 - learning_rate: 0.0010
Epoch 2/60
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6265 - loss: 0.8429 - val_accuracy: 0.2556 - val_loss: 2.5799 - learning_rate: 0.0010
Epoch 3/60
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6882 - loss: 0.6994 - val_accuracy: 0.2083 - val_loss: 5.0713 - learning_rate: 0.0010
Epoch 4/60
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7490 - loss: 0.5915 - val_accuracy: 0.1361 - val_loss: 3.4338 - learning_rate: 0.0010
Epoch 5/60
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7931 - loss: 0.4990 - val_accuracy: 0.1806 - val_loss: 2.8121 - learning_rate: 0.0010
Epoch 6/60
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7897 - loss: 0.4931 - val_accuracy: 

In [13]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split as tts

# carve a small validation slice out of train for early stopping
X_tr_h, X_val_h, y_tr_h, y_val_h = tts(
    X_train_hand_s, y_train, test_size=0.15, random_state=RANDOM_STATE, stratify=y_train
)

lgbm = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=n_classes,
    num_leaves=31,
    learning_rate=0.03,
    n_estimators=800,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

lgbm.fit(
    X_tr_h, y_tr_h,
    eval_set=[(X_val_h, y_val_h)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)],
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002717 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25500
[LightGBM] [Info] Number of data points in the train set: 2040, number of used features: 100
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.403947
[100]	valid_0's multi_logloss: 0.188991
[150]	valid_0's multi_logloss: 0.110753
[200]	valid_0's multi_logloss: 0.0819548
[250]	valid_0's multi_logloss: 0.0683913
[300]	valid_0's multi_logloss: 0.060845
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[350]	valid_0's multi_logloss: 0.0555374
[

,learning_rate,0.03
,n_estimators,800
,objective,'multiclass'
,class_weight,'balanced'
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,num_class,4
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1


In [14]:
from sklearn.metrics import accuracy_score

# CNN probs on the same validation slice used for LightGBM's early stopping
# (re-derive the matching mel-spectrogram rows by index)
_, val_pos, _, _ = tts(
    np.arange(len(X_train_hand_s)), y_train, test_size=0.15,
    random_state=RANDOM_STATE, stratify=y_train
)
X_val_mel = X_train_mel[val_pos]

cnn_val_proba = cnn.predict(X_val_mel, verbose=0)
lgbm_val_proba = lgbm.predict_proba(X_val_h)

best_w, best_acc = 0.5, 0.0
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * cnn_val_proba + (1 - w) * lgbm_val_proba
    acc = accuracy_score(y_val_h, blend.argmax(axis=1))
    if acc > best_acc:
        best_acc, best_w = acc, w

print(f"Best CNN weight on validation: {best_w:.2f}  (val acc {best_acc:.3f})")

Best CNN weight on validation: 0.40  (val acc 1.000)


In [15]:
cnn_test_proba = cnn.predict(X_test_mel, verbose=0)
lgbm_test_proba = lgbm.predict_proba(X_test_hand_s)

ensemble_proba = best_w * cnn_test_proba + (1 - best_w) * lgbm_test_proba
y_pred = ensemble_proba.argmax(axis=1)

# also report each model alone, so you can show the ensemble lift in your report
y_pred_cnn = cnn_test_proba.argmax(axis=1)
y_pred_lgbm = lgbm_test_proba.argmax(axis=1)

print(f"CNN-only test accuracy:      {accuracy_score(y_test_arr, y_pred_cnn):.3f}")
print(f"LightGBM-only test accuracy: {accuracy_score(y_test_arr, y_pred_lgbm):.3f}")
print(f"Ensemble test accuracy:      {accuracy_score(y_test_arr, y_pred):.3f}")

CNN-only test accuracy:      0.825
LightGBM-only test accuracy: 0.850
Ensemble test accuracy:      0.875


In [16]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

acc = accuracy_score(y_test_arr, y_pred)
prec = precision_score(y_test_arr, y_pred, average="macro", zero_division=0)
rec = recall_score(y_test_arr, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test_arr, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_test_arr, y_pred)

print(f"\nAccuracy:  {acc:.3f}")
print(f"Precision (macro): {prec:.3f}")
print(f"Recall (macro):    {rec:.3f}")
print(f"F1 Score (macro):  {f1:.3f}")
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nPer-class report:\n{classification_report(y_test_arr, y_pred, target_names=TARGET_CLASSES, zero_division=0)}")

os.makedirs("/content/reports/figures", exist_ok=True)

def plot_training_curves(history, filename):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history.history["accuracy"], label="train")
    axes[0].plot(history.history["val_accuracy"], label="val")
    axes[0].set_title("CNN Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history.history["loss"], label="train")
    axes[1].plot(history.history["val_loss"], label="val")
    axes[1].set_title("CNN Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    path = os.path.join("/content/reports/figures", filename)
    plt.savefig(path, format="jpg"); plt.close()
    print("Saved:", path)

def plot_audio_confusion_matrix(cm, class_names, filename):
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", cbar=True,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 11, "weight": "bold"}, ax=ax)
    ax.set_title("Confusion Matrix — Audio CNN+LightGBM Ensemble (Test Set)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    path = os.path.join("/content/reports/figures", filename)
    plt.savefig(path, format="jpg"); plt.close()
    print("Saved:", path)

def plot_metrics_bar(metrics_dict, filename):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(metrics_dict.keys(), metrics_dict.values(), color="#7B5EA7")
    ax.set_ylim(0, 1)
    for i, v in enumerate(metrics_dict.values()):
        ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold")
    ax.set_title("Ensemble Model — Evaluation Summary")
    plt.tight_layout()
    path = os.path.join("/content/reports/figures", filename)
    plt.savefig(path, format="jpg"); plt.close()
    print("Saved:", path)

plot_training_curves(history, "audio_training_curves.jpg")
plot_audio_confusion_matrix(cm, TARGET_CLASSES, "audio_confusion_matrix.jpg")
plot_metrics_bar({"Accuracy": acc, "Precision": prec, "Recall": rec, "F1 Score": f1},
                  "audio_metrics_summary.jpg")


Accuracy:  0.875
Precision (macro): 0.894
Recall (macro):    0.877
F1 Score (macro):  0.883

Confusion Matrix:
[[24  7  1  0]
 [ 2 37  1  0]
 [ 1  2 21  0]
 [ 1  0  0 23]]

Per-class report:
              precision    recall  f1-score   support

     silence       0.86      0.75      0.80        32
     ambient       0.80      0.93      0.86        40
       paper       0.91      0.88      0.89        24
        loud       1.00      0.96      0.98        24

    accuracy                           0.88       120
   macro avg       0.89      0.88      0.88       120
weighted avg       0.88      0.88      0.87       120

Saved: /content/reports/figures/audio_training_curves.jpg
Saved: /content/reports/figures/audio_confusion_matrix.jpg
Saved: /content/reports/figures/audio_metrics_summary.jpg


In [17]:
import pickle

save_dir = DRIVE_SAVE_DIR if DRIVE_SAVE_DIR else "/content/TrueWatch_models"
os.makedirs(save_dir, exist_ok=True)

cnn.save(os.path.join(save_dir, "audio_cnn.keras"))

with open(os.path.join(save_dir, "audio_lgbm.pkl"), "wb") as f:
    pickle.dump(lgbm, f)

with open(os.path.join(save_dir, "audio_scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os.path.join(save_dir, "audio_label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

with open(os.path.join(save_dir, "ensemble_weight.txt"), "w") as f:
    f.write(str(best_w))

!cp -r /content/reports {save_dir}/

print("Saved to:", save_dir)
print("Files:", os.listdir(save_dir))

Saved to: /content/drive/MyDrive/TrueWatch_models
Files: ['audio_cnn.keras', 'audio_lgbm.pkl', 'audio_scaler.pkl', 'audio_label_encoder.pkl', 'ensemble_weight.txt', 'reports']
